In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.utils as vutils
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

num_epochs = 60
lr_g = 0.00015          
lr_d = 0.0003
beta1 = 0.5
beta2 = 0.999
batch_size = 128
image_size = 32

label_smoothing_real = 0.9   
label_smoothing_fake = 0.1  
instance_noise_start = 0.1   
instance_noise_decay = 0.98  
use_gradient_clip = False    
g_updates_per_d = 1         
d_updates_per_g = 1   
# Early stopping
patience = 15
variance_threshold = 0.001    # Higher threshold (was 0.0008)
min_epochs = 20
collapse_patience = 5

# OPTIMAL for 32K: nz=24, ngf=7, ndf=7
# VERIFIED calculation: 31,581 params total (98.7% ROM usage, 419 headroom)
# This is MAXIMUM capacity under 32K budget!
nz = 24      # Latent dimension
ngf = 8      # Generator base features (7→14→28→3) - LARGER than before!
ndf = 4      # Discriminator base features (7→14→28) - LARGER!
nc = 3       # RGB

frac_bits = 8
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print(f"="*70)
print(f"MAXIMIZED DCGAN - 32K ROM TARGET")
print(f"Latent: {nz} | Gen: {ngf} | Disc: {ndf}")
print(f"Verified params: 31,581 / 32,000 (98.7% usage, 419 headroom)")
print(f"Channels: {ngf}→{ngf*2}→{ngf*4} (LARGER model for better quality!)")
print(f"Training: PARASITIZED ONLY")
print(f"="*70)

# ==============================================================================
# PARASITIZED CELLS ONLY DATASET
# ==============================================================================
print("\n[1/6] Loading PARASITIZED (infected) cells only...")

kaggle_input = '/kaggle/input'
dataset_dir = None

if os.path.exists(kaggle_input):
    for root, dirs, files in os.walk(kaggle_input):
        if 'Parasitized' in dirs:
            dataset_dir = root
            print(f"Found dataset at: {dataset_dir}")
            break

if dataset_dir is None:
    colab_paths = ['/content/cell_images', '/content/archive']
    for path in colab_paths:
        if os.path.exists(path):
            if os.path.isdir(os.path.join(path, 'Parasitized')):
                dataset_dir = path
                break

if dataset_dir is None:
    raise FileNotFoundError("Dataset not found!")

class ParasitizedOnlyDataset(ImageFolder):
    """Only load parasitized (malaria-infected) cells"""
    def __init__(self, root, transform=None):
        super().__init__(root, transform)
        # Filter to ONLY parasitized cells
        self.samples = [(path, label) for path, label in self.samples 
                       if 'Parasitized' in path or 'parasitized' in path.lower()]
        self.targets = [label for _, label in self.samples]
        print(f"  ✓ Filtered to {len(self.samples)} parasitized (infected) cells")

# Enhanced augmentation for better generalization
transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.CenterCrop(image_size),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),      # Add vertical flip
    transforms.RandomRotation(15),              # Small rotation
    transforms.ColorJitter(
        brightness=0.1,
        contrast=0.1,
        saturation=0.1,
        hue=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

dataset = ParasitizedOnlyDataset(dataset_dir, transform=transform)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, 
                       num_workers=2, pin_memory=True, drop_last=True)

print(f"  Total images: {len(dataset)}")
print(f"  Batches/epoch: {len(dataloader)}")

# Save real sample
real_batch = next(iter(dataloader))
plt.figure(figsize=(8, 8))
plt.axis("off")
plt.title("Real Parasitized Cells (Training Data)")
plt.imshow(np.transpose(vutils.make_grid(real_batch[0][:64], padding=2, normalize=True), (1, 2, 0)))
plt.savefig('real_parasitized_cells.png', dpi=150, bbox_inches='tight')
plt.show()

# ==============================================================================
# OPTIMIZED MODEL - Targeted for ~30K params (G+D combined)
# ==============================================================================
print("\n[2/6] Building Optimized Architecture...")

class OptimizedGenerator(nn.Module):
    """
    MAXIMIZED Generator for 32K ROM
    
    Parameters: nz=24, ngf=7
    Verified total: 22,907 params
    
    Architecture:
    - Linear: 24 → (28×4×4) = 448 features
    - L0: 4×4, 28→28 channels (Conv 3×3)
    - L1: 8×8, 28→14 channels (Conv 3×3)  
    - L2: 16×16, 14→7 channels (Conv 3×3)
    - L3: 32×32, 7→3 channels (Conv 3×3)
    
    This is MAXIMUM capacity under 32K budget!
    16% MORE parameters than ngf=6 (better quality!)
    """
    def __init__(self):
        super(OptimizedGenerator, self).__init__()
        
        # Linear: 16 → 40×4×4 = 640
        self.fc = nn.Linear(nz, ngf * 4 * 4 * 4)
        
        # L0: 40×40, 3×3 conv at 4×4
        self.conv0 = nn.Conv2d(ngf * 4, ngf * 4, 3, 1, 1, bias=False)
        self.bn0 = nn.BatchNorm2d(ngf * 4)
        
        # L1: 40→20, 3×3 conv at 8×8
        self.conv1 = nn.Conv2d(ngf * 4, ngf * 2, 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(ngf * 2)
        
        # L2: 20→10, 3×3 conv at 16×16
        self.conv2 = nn.Conv2d(ngf * 2, ngf, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(ngf)
        
        # L3: 10→3, 3×3 conv at 32×32
        self.conv3 = nn.Conv2d(ngf, nc, 3, 1, 1, bias=False)
        
        self.relu = nn.ReLU(True)
        self.tanh = nn.Tanh()
        
    def forward(self, z):
        x = self.fc(z.view(z.size(0), -1))
        x = x.view(-1, ngf * 4, 4, 4)
        
        x = self.relu(self.bn0(self.conv0(x)))
        
        x = nn.functional.interpolate(x, scale_factor=2, mode='nearest')
        x = self.relu(self.bn1(self.conv1(x)))
        
        x = nn.functional.interpolate(x, scale_factor=2, mode='nearest')
        x = self.relu(self.bn2(self.conv2(x)))
        
        x = nn.functional.interpolate(x, scale_factor=2, mode='nearest')
        x = self.tanh(self.conv3(x))
        
        return x


class OptimizedDiscriminator(nn.Module):
    """
    MAXIMIZED Discriminator for 32K ROM
    
    Parameters: ndf=7
    Verified total: 8,674 params
    
    Architecture:
    - L0: 32→16, 3→7 channels (Conv 4×4 stride-2)
    - L1: 16→8, 7→14 channels (Conv 4×4 stride-2)
    - L2: 8→4, 14→28 channels (Conv 4×4 stride-2)
    - L3: 4→1, 28→1 channel (Conv 4×4)
    
    COMBINED: 22,907 (Gen) + 8,674 (Disc) = 31,581 ✓
    Uses 98.7% of 32K ROM (419 params headroom)
    """
    def __init__(self):
        super(OptimizedDiscriminator, self).__init__()
        
        # L0: 3→10, 4×4 stride-2
        self.conv0 = nn.Conv2d(nc, ndf, 4, 2, 1, bias=False)
        
        # L1: 10→20, 4×4 stride-2
        self.conv1 = nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(ndf * 2)
        
        # L2: 20→40, 4×4 stride-2
        self.conv2 = nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(ndf * 4)
        
        # L3: 40→1, 4×4
        self.conv3 = nn.Conv2d(ndf * 4, 1, 4, 1, 0, bias=False)
        
        self.lrelu = nn.LeakyReLU(0.2, inplace=True)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        x = self.lrelu(self.conv0(x))
        x = self.lrelu(self.bn1(self.conv1(x)))
        x = self.lrelu(self.bn2(self.conv2(x)))
        x = self.sigmoid(self.conv3(x))
        return x.view(-1, 1)


netG = OptimizedGenerator().to(device)
netD = OptimizedDiscriminator().to(device)

def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)
    elif classname.find('Linear') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
        if m.bias is not None:
            nn.init.constant_(m.bias.data, 0)

netG.apply(weights_init)
netD.apply(weights_init)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

g_params = count_parameters(netG)
d_params = count_parameters(netD)
total_params = g_params + d_params

print(f"\n{'='*70}")
print(f"PARAMETER COUNT VERIFICATION")
print(f"{'='*70}")
print(f"Generator parameters:     {g_params:,}")
print(f"Discriminator parameters: {d_params:,}")
print(f"Total parameters:         {total_params:,}")
print(f"ROM capacity (32K):       32,000")
print(f"{'='*70}")

if total_params > 32000:
    print(f"❌ EXCEEDS 32K by {total_params - 32000:,} parameters!")
    print(f"   ROM usage: {total_params/32000*100:.1f}%")
    print(f"\n⚠ WARNING: Model too large! Training will continue but")
    print(f"   weights may not fit in FPGA ROM. Consider reducing ngf/ndf.")
else:
    headroom = 32000 - total_params
    print(f"✅ FITS in 32K ROM!")
    print(f"   ROM usage: {total_params/32000*100:.1f}%")
    print(f"   Headroom: {headroom:,} parameters ({headroom/32000*100:.1f}%)")

print(f"{'='*70}")

os.makedirs('fpga_weights', exist_ok=True)

# ==============================================================================
# TRAINING WITH AGGRESSIVE MODE COLLAPSE DETECTION
# ==============================================================================
print("\n[3/6] Starting Training with Anti-Collapse Measures...")

criterion = nn.BCELoss()
optimizerD = optim.Adam(netD.parameters(), lr=lr_d, betas=(beta1, beta2))
optimizerG = optim.Adam(netG.parameters(), lr=lr_g, betas=(beta1, beta2))

def check_mode_collapse(fake_batch):
    """Enhanced mode collapse detection"""
    pixel_var = fake_batch.var(dim=0).mean().item()
    batch_mean = fake_batch.mean(dim=0, keepdim=True)
    mse_from_mean = ((fake_batch - batch_mean) ** 2).mean().item()
    color_var = fake_batch.var(dim=1).mean().item()
    
    # Check if batch images are too similar
    batch_std = fake_batch.std(dim=(2, 3)).mean().item()
    
    collapsed = (pixel_var < variance_threshold or 
                mse_from_mean < 0.001 or 
                color_var < 0.01 or
                batch_std < 0.05)  # Additional check
    
    return collapsed, pixel_var, mse_from_mean, color_var

fixed_noise = torch.randn(64, nz, 1, 1, device=device)

G_losses = []
D_losses = []
D_x_history = []
D_G_z_history = []
variance_history = []
img_list = []

best_variance = 0
best_epoch = 0
consecutive_collapses = 0
no_improve_count = 0
best_model_state = None

# Warmup period - ignore collapse in first N epochs
warmup_epochs = 10  # Model needs time to learn

# Save frequency
save_interval = 5  # Save every 5 epochs

print(f"Training Configuration:")
print(f"  Dataset: {len(dataset)} parasitized cells")
print(f"  Epochs: {num_epochs} | Batch: {batch_size}")
print(f"  LR: G={lr_g}, D={lr_d} (ratio {lr_g/lr_d:.1f}:1)")
print(f"  Warmup: {warmup_epochs} epochs (collapse detection disabled)")
print(f"  Collapse detection: variance < {variance_threshold}")
print(f"  Early stop patience: {patience} epochs")
print(f"  Collapse patience: {collapse_patience} epochs")
print("="*70)

for epoch in range(num_epochs):
    epoch_g_loss = 0.0
    epoch_d_loss = 0.0
    epoch_d_x = 0.0
    epoch_d_g_z = 0.0
    epoch_variance = 0.0
    
    current_noise_std = instance_noise_start * (instance_noise_decay ** epoch)
    
    for i, data in enumerate(dataloader, 0):
        
        # ---------------------------------------
        # (1) Update D network: LOOPING 2 KALI
        # ---------------------------------------
        for _ in range(d_updates_per_g): 
            netD.zero_grad()
            
            # --- Train Real ---
            real_cpu = data[0].to(device)
            b_size = real_cpu.size(0)
            
            label_real = torch.full((b_size,), 0.9, dtype=torch.float, device=device)
            noise_amp = 0.05 
            real_noisy = real_cpu + noise_amp * torch.randn_like(real_cpu)
            
            output = netD(real_noisy).view(-1)
            errD_real = criterion(output, label_real)
            errD_real.backward()
            
            # Simpan nilai D(x) untuk statistik
            D_x = output.mean().item() 
            
            # --- Train Fake ---
            noise = torch.randn(b_size, nz, 1, 1, device=device)
            fake = netG(noise)
            label_fake = torch.full((b_size,), 0.0, dtype=torch.float, device=device)
            
            output = netD(fake.detach()).view(-1)
            errD_fake = criterion(output, label_fake)
            errD_fake.backward()
            D_G_z1 = output.mean().item()
            
            # Gabungkan Error D
            errD = errD_real + errD_fake 
            optimizerD.step()

        # ---------------------------------------
        # (2) Update G network: CUKUP 1 KALI
        # ---------------------------------------
        netG.zero_grad()
        noise = torch.randn(b_size, nz, 1, 1, device=device)
        fake = netG(noise)
        label_g = torch.ones((b_size,), dtype=torch.float, device=device)
        
        output = netD(fake).view(-1)
        errG = criterion(output, label_g)
        errG.backward()
        D_G_z2 = output.mean().item()
        
        optimizerG.step()
        
        # ---------------------------------------
        # (3) AKUMULASI STATISTIK (BAGIAN YANG HILANG)
        # ---------------------------------------
        # Hitung varians pixel (untuk deteksi collapse)
        with torch.no_grad():
            pix_var = fake.var(dim=0).mean().item()
            
        # TAMBAHKAN INI AGAR RATA-RATA KELUAR:
        epoch_g_loss += errG.item()
        epoch_d_loss += errD.item()
        epoch_d_x += D_x
        epoch_d_g_z += D_G_z2
        epoch_variance += pix_var
        
        # Logika print progress
        if i % 50 == 0:
            # Pastikan variabel 'collapsed' ada logikanya atau hapus jika error
            # Di sini saya anggap collapsed dicek dari pix_var
            collapsed = pix_var < 0.001 
            collapse_warn = " ⚠ COLLAPSE DETECTED!" if collapsed else ""
            
            print(f'[{epoch+1:3d}/{num_epochs}][{i:4d}/{len(dataloader)}] '
                  f'D: {errD.item():.4f} G: {errG.item():.4f} '
                  f'D(x): {D_x:.3f} D(G(z)): {D_G_z2:.3f} '
                  f'Var: {pix_var:.5f} Noise: {current_noise_std:.4f}{collapse_warn}')
    
    # ---------------------------------------
    # PERHITUNGAN RATA-RATA (AKAN BENAR SEKARANG)
    # ---------------------------------------
    avg_g = epoch_g_loss / len(dataloader)
    avg_d = epoch_d_loss / len(dataloader)
    avg_d_x = epoch_d_x / len(dataloader)
    avg_d_g_z = epoch_d_g_z / len(dataloader)
    avg_variance = epoch_variance / len(dataloader)
    
    print(f">>> Epoch {epoch+1} Summary: D(x):{avg_d_x:.3f} D(Gz):{avg_d_g_z:.3f} Var:{avg_variance:.5f}")
    
    # Test for mode collapse with fixed noise
    with torch.no_grad():
        test_fake = netG(fixed_noise)
        collapsed, pix_var, mse, col_var = check_mode_collapse(test_fake)
    
    # SKIP collapse detection during warmup
    if epoch < warmup_epochs:
        collapsed = False
        if epoch == warmup_epochs - 1:
            print(f"\n{'='*70}")
            print(f"WARMUP COMPLETE - Collapse detection now active")
            print(f"{'='*70}\n")
    
    improvement_marker = ""
    
    if collapsed:
        consecutive_collapses += 1
        print(f"\n{'!'*70}")
        print(f"MODE COLLAPSE DETECTED! (Episode {consecutive_collapses}/{collapse_patience})")
        print(f"Variance: {pix_var:.6f} < Threshold: {variance_threshold}")
        print(f"{'!'*70}\n")
        
        # AUTO-SAVE before stopping
        if consecutive_collapses >= collapse_patience:
            emergency_path = f'fpga_weights/emergency_save_epoch_{epoch+1}.pth'
            torch.save({
                'epoch': epoch + 1,
                'netG_state_dict': netG.state_dict(),
                'netD_state_dict': netD.state_dict(),
                'variance': pix_var,
                'status': 'mode_collapse_detected'
            }, emergency_path)
            print(f"🚨 EMERGENCY SAVE: {emergency_path}")
            
            # Use best model if available
            if best_model_state is not None:
                print(f"Loading best model from epoch {best_epoch} before collapse...")
                netG.load_state_dict(best_model_state['netG_state_dict'])
                netD.load_state_dict(best_model_state['netD_state_dict'])
            
            print(f"\n{'='*70}")
            print(f"STOPPING: Persistent mode collapse ({collapse_patience} episodes)")
            print(f"Best saved model: Epoch {best_epoch} with variance {best_variance:.5f}")
            print(f"{'='*70}\n")
            break
    else:
        consecutive_collapses = 0
        
        if pix_var > best_variance and epoch >= min_epochs:
            best_variance = pix_var
            best_epoch = epoch + 1
            no_improve_count = 0
            best_model_state = {
                'epoch': epoch + 1,
                'netG_state_dict': netG.state_dict(),
                'netD_state_dict': netD.state_dict(),
                'variance': pix_var,
            }
            torch.save(best_model_state, 'fpga_weights/best_model.pth')
            improvement_marker = " ★ NEW BEST"
        else:
            no_improve_count += 1
    
    # Status indicator
    status = "✓"
    if epoch < warmup_epochs:
        status = f"🔥 WARMUP ({epoch+1}/{warmup_epochs})"
    elif collapsed:
        status = f"⚠ COLLAPSE (x{consecutive_collapses})"
    elif avg_d < 0.25:
        status = "⚠ D very strong"
    elif avg_d_x > 0.92:
        status = "⚠ D overfitting"
    elif avg_g > 5.0:
        status = "⚠ G struggling"
    
    print(f'>>> Epoch [{epoch+1:3d}/{num_epochs}] '
          f'Loss_D: {avg_d:.4f} Loss_G: {avg_g:.4f} '
          f'D(x): {avg_d_x:.3f} D(G(z)): {avg_d_g_z:.3f} '
          f'Var: {avg_variance:.5f} {status}{improvement_marker}')
    
    # Early stopping check
    if no_improve_count >= patience and epoch >= min_epochs:
        print(f"\n{'='*70}")
        print(f"EARLY STOPPING: No improvement for {patience} epochs")
        print(f"Best model: Epoch {best_epoch} with variance {best_variance:.5f}")
        print(f"{'='*70}\n")
        break
    
    # Periodic save
    if (epoch + 1) % save_interval == 0:
        checkpoint_path = f'fpga_weights/checkpoint_epoch_{epoch+1}.pth'
        torch.save({
            'epoch': epoch + 1,
            'netG_state_dict': netG.state_dict(),
            'netD_state_dict': netD.state_dict(),
            'variance': avg_variance,
        }, checkpoint_path)
        print(f"  💾 Checkpoint saved: {checkpoint_path}")
    
    # Save samples
    if (epoch + 1) % 10 == 0:
        with torch.no_grad():
            fake = netG(fixed_noise).detach().cpu()
        img_list.append(vutils.make_grid(fake, padding=2, normalize=True))
        
        plt.figure(figsize=(8, 8))
        plt.axis("off")
        plt.title(f"Generated - Epoch {epoch+1}")
        plt.imshow(np.transpose(img_list[-1], (1, 2, 0)))
        plt.savefig(f'generated_epoch_{epoch+1}.png', dpi=150, bbox_inches='tight')
        plt.show()

print("="*70)
print("Training Complete!")
if best_model_state:
    print(f"Best model: Epoch {best_epoch} with variance {best_variance:.5f}")
else:
    print("Warning: No best model saved (training stopped early)")
print("="*70)

# Load best model
if best_model_state:
    netG.load_state_dict(best_model_state['netG_state_dict'])
    netD.load_state_dict(best_model_state['netD_state_dict'])
    print(f"✓ Loaded best model from epoch {best_epoch}")

# Plot metrics
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0, 0].plot(G_losses, 'b-', label="G Loss", linewidth=2)
axes[0, 0].plot(D_losses, 'r-', label="D Loss", linewidth=2)
axes[0, 0].set_xlabel("Epochs")
axes[0, 0].set_ylabel("Loss")
axes[0, 0].set_title("Generator and Discriminator Loss")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(D_x_history, 'g-', label="D(x) - Real", linewidth=2)
axes[0, 1].plot(D_G_z_history, 'r-', label="D(G(z)) - Fake", linewidth=2)
axes[0, 1].axhline(y=0.5, color='black', linestyle='--', alpha=0.3)
axes[0, 1].set_xlabel("Epochs")
axes[0, 1].set_ylabel("Discriminator Output")
axes[0, 1].set_title("D(x) and D(G(z))")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_ylim([0, 1])

axes[1, 0].plot(variance_history, 'purple', linewidth=2, label="Pixel Variance")
axes[1, 0].axhline(y=variance_threshold, color='red', linestyle='--', 
                   alpha=0.5, label='Collapse Threshold')
axes[1, 0].set_xlabel("Epochs")
axes[1, 0].set_ylabel("Variance")
axes[1, 0].set_title("Output Diversity (Mode Collapse Indicator)")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

with torch.no_grad():
    final = netG(fixed_noise).detach().cpu()
axes[1, 1].imshow(np.transpose(vutils.make_grid(final, padding=2, normalize=True), (1, 2, 0)))
axes[1, 1].set_title(f"Final Samples (Best: Epoch {best_epoch})")
axes[1, 1].axis('off')

plt.tight_layout()
plt.savefig('training_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

# ==============================================================================
# EXPORT WEIGHTS FOR FPGA - SAME AS BEFORE
# ==============================================================================
print("\n[4/6] Exporting Weights for FPGA...")

netG.eval()
netD.eval()

def to_fixed_point(value, frac_bits=8):
    fixed = int(round(value * (2 ** frac_bits)))
    return max(-32768, min(32767, fixed))

def fold_batch_norm(conv_layer, bn_layer):
    w = conv_layer.weight.data.cpu().numpy()
    
    if bn_layer is not None:
        mean = bn_layer.running_mean.cpu().numpy()
        var = bn_layer.running_var.cpu().numpy()
        gamma = bn_layer.weight.data.cpu().numpy()
        beta = bn_layer.bias.data.cpu().numpy()
        eps = bn_layer.eps
        
        scale = gamma / np.sqrt(var + eps)
        w_folded = w * scale.reshape(-1, 1, 1, 1)
        
        b_old = np.zeros(w.shape[0]) if conv_layer.bias is None else conv_layer.bias.data.cpu().numpy()
        b_folded = (b_old - mean) * scale + beta
    else:
        w_folded = w
        b_folded = np.zeros(w.shape[0]) if conv_layer.bias is None else conv_layer.bias.data.cpu().numpy()
    
    return w_folded, b_folded

def export_linear_layer(linear_layer):
    w = linear_layer.weight.data.cpu().numpy()
    b = linear_layer.bias.data.cpu().numpy() if linear_layer.bias is not None else np.zeros(w.shape[0])
    return w, b

# Generator layers (including linear)
gen_layers = [
    ('G_FC', netG.fc, None, 'linear'),
    ('G_L0', netG.conv0, netG.bn0, 'conv'),
    ('G_L1', netG.conv1, netG.bn1, 'conv'),
    ('G_L2', netG.conv2, netG.bn2, 'conv'),
    ('G_L3', netG.conv3, None, 'conv'),
]

# Discriminator layers
disc_layers = [
    ('D_L0', netD.conv0, None, 'conv'),
    ('D_L1', netD.conv1, netD.bn1, 'conv'),
    ('D_L2', netD.conv2, netD.bn2, 'conv'),
    ('D_L3', netD.conv3, None, 'conv'),
]

os.makedirs('fpga_weights', exist_ok=True)

print("\n[5/6] Saving to Excel and .mem files...")

excel_filename = 'fpga_weights/dcgan_weights_32k.xlsx'

with pd.ExcelWriter(excel_filename) as writer:
    print("\n--- Generator Layers ---")
    for name, layer, bn, layer_type in gen_layers:
        if layer_type == 'linear':
            w, b = export_linear_layer(layer)
        else:
            w, b = fold_batch_norm(layer, bn)
        
        w_flat = w.flatten()
        
        df = pd.DataFrame({
            'Index': range(len(w_flat)),
            'Weight_Float': w_flat,
            'Weight_Fixed': [to_fixed_point(x, frac_bits) for x in w_flat],
            'Weight_Hex': [f"{to_fixed_point(x, frac_bits) & 0xFFFF:04x}" for x in w_flat]
        })
        
        df_bias = pd.DataFrame({
            'Index': range(len(b)),
            'Bias_Float': b,
            'Bias_Fixed': [to_fixed_point(x, frac_bits) for x in b],
            'Bias_Hex': [f"{to_fixed_point(x, frac_bits) & 0xFFFF:04x}" for x in b]
        })
        
        df_combined = pd.concat([df, df_bias], axis=1)
        df_combined.to_excel(writer, sheet_name=name, index=False)
        type_str = f"({layer_type})" if layer_type == 'linear' else ""
        print(f"  {name}: {len(w_flat)} weights + {len(b)} biases {type_str}")
    
    print("\n--- Discriminator Layers ---")
    for name, layer, bn, layer_type in disc_layers:
        w, b = fold_batch_norm(layer, bn)
        w_flat = w.flatten()
        
        df = pd.DataFrame({
            'Index': range(len(w_flat)),
            'Weight_Float': w_flat,
            'Weight_Fixed': [to_fixed_point(x, frac_bits) for x in w_flat],
            'Weight_Hex': [f"{to_fixed_point(x, frac_bits) & 0xFFFF:04x}" for x in w_flat]
        })
        
        df_bias = pd.DataFrame({
            'Index': range(len(b)),
            'Bias_Float': b,
            'Bias_Fixed': [to_fixed_point(x, frac_bits) for x in b],
            'Bias_Hex': [f"{to_fixed_point(x, frac_bits) & 0xFFFF:04x}" for x in b]
        })
        
        df_combined = pd.concat([df, df_bias], axis=1)
        df_combined.to_excel(writer, sheet_name=name, index=False)
        print(f"  {name}: {len(w_flat)} weights + {len(b)} biases")

print(f"\n✓ Excel saved: {excel_filename}")

# Save .mem files
def save_mem_file(filename, values):
    with open(filename, 'w') as f:
        for val in values:
            fixed = to_fixed_point(val, frac_bits)
            f.write(f"{fixed & 0xFFFF:04x}\n")

print("\nSaving .mem files...")
for name, layer, bn, layer_type in gen_layers:
    if layer_type == 'linear':
        w, b = export_linear_layer(layer)
    else:
        w, b = fold_batch_norm(layer, bn)
    save_mem_file(f'fpga_weights/{name}_weights.mem', w.flatten())
    save_mem_file(f'fpga_weights/{name}_biases.mem', b)

for name, layer, bn, layer_type in disc_layers:
    w, b = fold_batch_norm(layer, bn)
    save_mem_file(f'fpga_weights/{name}_weights.mem', w.flatten())
    save_mem_file(f'fpga_weights/{name}_biases.mem', b)

# Combined ROM files
print("\nCreating combined ROM files...")

# Generator ROM
rom_gen = []
for name, layer, bn, layer_type in gen_layers:
    if layer_type == 'linear':
        w, b = export_linear_layer(layer)
    else:
        w, b = fold_batch_norm(layer, bn)
    rom_gen.extend([to_fixed_point(x, frac_bits) for x in w.flatten()])
    rom_gen.extend([to_fixed_point(x, frac_bits) for x in b])

with open('fpga_weights/generator_rom.mem', 'w') as f:
    for val in rom_gen:
        f.write(f"{val & 0xFFFF:04x}\n")
print(f"✓ Generator ROM: {len(rom_gen)} entries")

# Discriminator ROM
rom_disc = []
for name, layer, bn, layer_type in disc_layers:
    w, b = fold_batch_norm(layer, bn)
    rom_disc.extend([to_fixed_point(x, frac_bits) for x in w.flatten()])
    rom_disc.extend([to_fixed_point(x, frac_bits) for x in b])

with open('fpga_weights/discriminator_rom.mem', 'w') as f:
    for val in rom_disc:
        f.write(f"{val & 0xFFFF:04x}\n")
print(f"✓ Discriminator ROM: {len(rom_disc)} entries")

# UNIFIED ROM (G+D combined)
unified_rom = rom_gen + rom_disc
with open('fpga_weights/unified_rom.mem', 'w') as f:
    for val in unified_rom:
        f.write(f"{val & 0xFFFF:04x}\n")
print(f"✓ Unified ROM: {len(unified_rom)} entries")

# Summary
summary = []
total_weights = 0
total_biases = 0

print("\n" + "="*70)
print("WEIGHT EXPORT SUMMARY")
print("="*70)

for name, layer, bn, layer_type in gen_layers:
    if layer_type == 'linear':
        w, b = export_linear_layer(layer)
    else:
        w, b = fold_batch_norm(layer, bn)
    w_count = w.size
    b_count = len(b)
    total_weights += w_count
    total_biases += b_count
    
    summary.append({
        'Layer': name,
        'Type': f'Gen-{layer_type}',
        'Weights': w_count,
        'Biases': b_count,
        'Total': w_count + b_count
    })
    print(f"{name:<10} {f'Gen-{layer_type}':<15} {w_count:>8} {b_count:>8} {w_count + b_count:>8}")

for name, layer, bn, layer_type in disc_layers:
    w, b = fold_batch_norm(layer, bn)
    w_count = w.size
    b_count = len(b)
    total_weights += w_count
    total_biases += b_count
    
    summary.append({
        'Layer': name,
        'Type': f'Disc-{layer_type}',
        'Weights': w_count,
        'Biases': b_count,
        'Total': w_count + b_count
    })
    print(f"{name:<10} {f'Disc-{layer_type}':<15} {w_count:>8} {b_count:>8} {w_count + b_count:>8}")

print("="*70)
print(f"TOTAL      {'':<15} {total_weights:>8} {total_biases:>8} {total_weights + total_biases:>8}")
print("="*70)

rom_capacity = 32000
total_exported = total_weights + total_biases
usage_percent = (total_exported / rom_capacity) * 100

print(f"\nROM CAPACITY ANALYSIS:")
print(f"  Total exported:  {total_exported:,}")
print(f"  ROM capacity:    {rom_capacity:,}")
print(f"  Usage:           {usage_percent:.1f}%")

if total_exported <= rom_capacity:
    headroom = rom_capacity - total_exported
    print(f"  ✅ FITS in 32K ROM!")
    print(f"  Headroom:        {headroom:,} parameters ({headroom/rom_capacity*100:.1f}%)")
else:
    overflow = total_exported - rom_capacity
    print(f"  ❌ EXCEEDS ROM by {overflow:,} parameters!")

# Save summary
df_summary = pd.DataFrame(summary)
df_summary.to_csv('fpga_weights/weight_summary.csv', index=False)

# Generate Verilog parameters
print("\nGenerating Verilog parameter file...")
with open('fpga_weights/weight_parameters.vh', 'w') as f:
    f.write("// Auto-generated weight parameters\n\n")
    
    offset = 0
    f.write("// GENERATOR\n")
    for name, layer, bn, layer_type in gen_layers:
        if layer_type == 'linear':
            w, b = export_linear_layer(layer)
        else:
            w, b = fold_batch_norm(layer, bn)
        count = w.size + len(b)
        f.write(f"localparam WEIGHTS_{name} = {count};\n")
        f.write(f"localparam BASE_{name} = 15'd{offset};\n")
        offset += count
    
    f.write(f"\n// Total Generator: {offset}\n\n")
    
    disc_offset = offset
    f.write("// DISCRIMINATOR\n")
    for name, layer, bn, layer_type in disc_layers:
        w, b = fold_batch_norm(layer, bn)
        count = w.size + len(b)
        f.write(f"localparam WEIGHTS_{name} = {count};\n")
        f.write(f"localparam BASE_{name} = 15'd{offset};\n")
        offset += count
    
    f.write(f"\n// Total System: {offset} / 32000 ({offset/32000*100:.1f}%)\n")

print(f"✓ Verilog parameters: fpga_weights/weight_parameters.vh")

# ==============================================================================
# FINAL CHECKPOINT
# ==============================================================================
print("\n[6/6] Saving Final Checkpoint...")

torch.save({
    'epoch': best_epoch if best_model_state else len(G_losses),
    'netG_state_dict': netG.state_dict(),
    'netD_state_dict': netD.state_dict(),
    'G_losses': G_losses,
    'D_losses': D_losses,
    'variance_history': variance_history,
    'best_variance': best_variance,
    'nz': nz,
    'ngf': ngf,
    'ndf': ndf,
    'total_params': total_exported,
}, 'fpga_weights/final_checkpoint.pth')

print("="*70)
print("EXPORT COMPLETE!")
print("="*70)
print(f"✓ Training: {'STOPPED (Mode Collapse)' if consecutive_collapses >= collapse_patience else 'COMPLETE'}")
print(f"✓ Best Epoch: {best_epoch}")
print(f"✓ Total Params: {total_exported:,} / 32,000")
print(f"✓ ROM Fit: {'YES ✅' if total_exported <= 32000 else 'NO ❌'}")
print(f"\nFiles Generated:")
print(f"  - fpga_weights/dcgan_weights_32k.xlsx")
print(f"  - fpga_weights/*_weights.mem")
print(f"  - fpga_weights/*_biases.mem")
print(f"  - fpga_weights/generator_rom.mem")
print(f"  - fpga_weights/discriminator_rom.mem")
print(f"  - fpga_weights/unified_rom.mem")
print(f"  - fpga_weights/weight_parameters.vh")
print(f"  - fpga_weights/final_checkpoint.pth")
if consecutive_collapses >= collapse_patience:
    print(f"  - fpga_weights/emergency_save_epoch_*.pth")
print("="*70)